# Data Cleaning and Merging Workflow

This notebook documents the step-by-step process used to clean and merge the restaurant datasets.

In [ ]:
import pandas as pd
import numpy as np
import os
import re

# Paths
DATA_SOURCE_DIR = r"../../Data/Data Sources"
OUTPUT_DIR = r"../../Data/Data Wrangling"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print("Setup complete. Data directory:", DATA_SOURCE_DIR)

## 1. Clean Transaction Data (dim_v2)
We aggregate transaction data to the restaurant level.

In [ ]:
print("Loading and cleaning dim_v2.csv...")
df = pd.read_csv(os.path.join(DATA_SOURCE_DIR, "dim_v2.csv"), low_memory=False)

# Date conversion
date_cols = ['date', 'created_at']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Derived columns
df['total_guests'] = df['adult'] + df['kids']

agg_funcs = {
    'revenue': 'sum',
    'id': 'count', # Total bookings
    'total_guests': 'sum',
    'no_show': 'sum',
    'is_temporary': 'sum' 
}

# Group by restaurant_id
restaurant_stats = df.groupby('restaurant_id').agg(agg_funcs).reset_index()
restaurant_stats.rename(columns={
    'revenue': 'total_revenue',
    'id': 'total_bookings',
    'no_show': 'total_no_shows',
    'is_temporary': 'total_temporary_bookings'
}, inplace=True)

# Calculate rates
restaurant_stats['avg_revenue_per_booking'] = restaurant_stats['total_revenue'] / restaurant_stats['total_bookings']
restaurant_stats['no_show_rate'] = restaurant_stats['total_no_shows'] / restaurant_stats['total_bookings']

print(f"Processed transactions for {len(restaurant_stats)} restaurants.")
trans_df = restaurant_stats
trans_df.head()

## 2. Clean Metadata (Tags, Bookings, Ratings)
We consolidate restaurant names, cuisines, facilities, and ratings.

In [ ]:
print("Loading and cleaning metadata files...")

# Dimensions Tags
tags_df = pd.read_csv(os.path.join(DATA_SOURCE_DIR, "dim_tags.csv"))
tags_df.drop_duplicates(subset=['restaurant_id'], inplace=True)
tags_df.fillna("Unknown", inplace=True)

# Booking Dimensions (Names)
bookings_df = pd.read_csv(os.path.join(DATA_SOURCE_DIR, "dim_bookings.csv"))
bookings_df = bookings_df[['restaurant_id', 'name', 'days_in_advance']]

# Customer Rating
ratings_df = pd.read_csv(os.path.join(DATA_SOURCE_DIR, "dim_customer_rating.csv"))

# Merge metadata
meta = pd.merge(bookings_df, tags_df, on='restaurant_id', how='left')
meta = pd.merge(meta, ratings_df, on='restaurant_id', how='left')

print(f"Processed metadata for {len(meta)} restaurants.")
meta_df = meta
meta_df.head()

## 3. Clean KOL Data
We aggregate KOL performance metrics (views, likes, comments) and count unique KOLs per restaurant.

In [ ]:
print("Loading and cleaning KOL data...")

posts = pd.read_csv(os.path.join(DATA_SOURCE_DIR, "KOL DATA - SMU - Posts.csv"))

def clean_num(x):
    if pd.isna(x): return 0
    if isinstance(x, (int, float)): return x
    x = str(x).replace(',', '')
    if 'k' in x.lower():
        try:
            return float(x.lower().replace('k', '')) * 1000
        except:
            pass
    try:
        return float(x)
    except:
        return 0

metric_cols = ['Views', 'Likes', 'Comments']
for col in metric_cols:
    posts[col] = posts[col].apply(clean_num)

posts['restaurant_id'] = pd.to_numeric(posts['Restaurant Code'], errors='coerce')

# Aggregate metrics
kol_stats = posts.groupby('restaurant_id')[metric_cols].sum().reset_index()
kol_stats.columns = ['restaurant_id', 'kol_views', 'kol_likes', 'kol_comments']

# Count unique KOLs
kol_count = posts.groupby('restaurant_id')['KOL_ID'].nunique().reset_index(name='unique_kols_used')

kol_final = pd.merge(kol_stats, kol_count, on='restaurant_id', how='left')

print(f"Processed KOL data for {len(kol_final)} restaurants.")
kol_df = kol_final
kol_df.head()

## 4. Clean Analytics Data
We attempt to link web analytics data to restaurants by parsing the `pagePath`.

In [ ]:
print("Loading and cleaning Analytics data...")
analytics_df = pd.read_csv(os.path.join(DATA_SOURCE_DIR, "analytics_data (1).csv"))

# Helper to map slugs
try:
    r_db = pd.read_csv(os.path.join(DATA_SOURCE_DIR, "KOL DATA - SMU - Restaurant DB All.csv"))
    
    def extract_slug(url):
        if pd.isna(url): return None
        return url.strip().split('/')[-1]
        
    r_db['slug'] = r_db['link'].apply(extract_slug)
    slug_map = r_db.set_index('slug')['ID'].to_dict()
    
    def extract_path_slug(path):
        if pd.isna(path): return None
        if 'restaurants/' in path:
            parts = path.split('restaurants/')
            if len(parts) > 1:
                return parts[1].split('/')[0].split('?')[0]
        return None
        
    analytics_df['slug'] = analytics_df['pagePath'].apply(extract_path_slug)
    analytics_df['restaurant_id'] = analytics_df['slug'].map(slug_map)
    
    matched = analytics_df.dropna(subset=['restaurant_id']).copy()
    
    analytics_agg = matched.groupby('restaurant_id').agg({
        'screenPageViews': 'sum',
        'purchaseRevenue': 'sum'
    }).reset_index()
    
    analytics_agg.rename(columns={
        'screenPageViews': 'web_views',
        'purchaseRevenue': 'web_revenue'
    }, inplace=True)
    
    print(f"Processed Analytics for {len(analytics_agg)} restaurants.")
except Exception as e:
    print(f"Error processing analytics: {e}")
    analytics_agg = pd.DataFrame()

analytics_df = analytics_agg
analytics_df.head()

## 5. Merge and Save Master Table
We now collect ALL unique restaurant IDs from every source to ensure no restaurant is left behind, then left join details onto this master list.

In [ ]:
# 1. Get all unique restaurant IDs
all_ids = set()
if 'restaurant_id' in trans_df.columns:
    all_ids.update(trans_df['restaurant_id'].dropna().unique())
if 'restaurant_id' in meta_df.columns:
    all_ids.update(meta_df['restaurant_id'].dropna().unique())
if 'restaurant_id' in kol_df.columns:
    all_ids.update(kol_df['restaurant_id'].dropna().unique())
if 'restaurant_id' in analytics_df.columns:
    all_ids.update(analytics_df['restaurant_id'].dropna().unique())
    
print(f"Total unique restaurants found across all sources: {len(all_ids)}")

# Create master dataframe with all IDs
master = pd.DataFrame({'restaurant_id': list(all_ids)})

# 2. Merge Data
print("Merging datasets...")

# Merge Metadata (Names, etc.)
master = pd.merge(master, meta_df, on='restaurant_id', how='left')

# Merge Transactions
master = pd.merge(master, trans_df, on='restaurant_id', how='left')

# Merge KOL
master = pd.merge(master, kol_df, on='restaurant_id', how='left')

# Merge Analytics
master = pd.merge(master, analytics_df, on='restaurant_id', how='left')

# Fill NaNs with 0 for metrics
metric_cols = [
    'total_revenue', 'total_bookings', 'total_guests', 
    'total_no_shows', 'total_temporary_bookings',
    'kol_views', 'kol_likes', 'kol_comments', 'unique_kols_used',
    'web_views', 'web_revenue'
]
for col in metric_cols:
    if col in master.columns:
        master[col] = master[col].fillna(0)

print(f"Total Master Rows: {len(master)}")

# Save
output_path = os.path.join(OUTPUT_DIR, "master_restaurant_table.csv")
master.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")